# Field Scan : entraînement du modèle maladies

Classification de photos de feuilles de bananier en 3 classes (saine, Black Sigatoka, Fusarium), avec une réponse « inconnu » quand le modèle n'est pas assez sûr de lui.

| Rôle | Jeu | Pourquoi |
| --- | --- | --- |
| Entraînement | Tanzanie 2023 | le modèle apprend |
| Validation | Tanzanie 2025 | choix de la meilleure époque et du seuil « inconnu » |
| Test | Bangladesh | autre pays ; cordana et pestalotiopsis, jamais vues, doivent donner « inconnu » |

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU T4**.

Données attendues sur Google Drive : `MyDrive/field_scan/feuilles.zip`, qui contient `images/`, `images.csv` et `doublons_bangladesh.txt` (produits par `outils/preparer_feuilles.py`).

In [ ]:
# 1. Accès au Google Drive et décompression des images
from google.colab import drive
drive.mount('/content/drive')

DOSSIER_DRIVE = '/content/drive/MyDrive/field_scan'
!unzip -q -o "{DOSSIER_DRIVE}/feuilles.zip" -d /content/
!ls /content/feuilles

In [ ]:
# 2. Réglages
import random, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch

RACINE = Path('/content/feuilles')
CLASSES = ['saine', 'black_sigatoka', 'fusarium']   # les 3 classes apprises
EPOQUES = 8
TAILLE_LOT = 64
TAUX_APPRENTISSAGE = 3e-4
GRAINE = 0
ACCEPTATION_VALIDATION = 0.95   # le seuil « inconnu » garde 95 % des images de validation

random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
appareil = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Calcul sur :', appareil)

In [ ]:
# 3. Découpage : par pays et par campagne, jamais au hasard
images = pd.read_csv(RACINE / 'images.csv')
# Images comparées sans leur extension (.jpg ou .jpeg)
exclus = {l.strip().rsplit('.', 1)[0] for l in open(RACINE / 'doublons_bangladesh.txt', encoding='utf-8')
          if l.strip() and not l.startswith('#')}

def source_courte(chemin):
    # 'dossier/.../OriginalSet/healthy/122.jpeg' -> 'healthy/122'
    return '/'.join(chemin.split('/')[-2:]).rsplit('.', 1)[0]

entrainement = images[images.jeu == 'tanzanie_2023']
validation = images[images.jeu == 'tanzanie_2025']
test = images[(images.jeu == 'bangladesh') & ~images.source.map(source_courte).isin(exclus)]

for nom, df in [('entraînement', entrainement), ('validation', validation), ('test', test)]:
    print(f'{nom:13s} {len(df):6d} images  ', df.classe.value_counts().to_dict())
assert len(test) == 893, 'le test doit compter 893 images après retrait des doublons'

In [ ]:
# 4. Chargement des images, avec augmentations pour l'entraînement seulement
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

NORMALISATION = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # celle d'ImageNet
aug_entrainement = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
    T.ToTensor(), NORMALISATION,
])
aug_evaluation = T.Compose([T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    def __init__(self, df, transformation):
        self.chemins = list(df.image)
        self.etiquettes = [CLASSES.index(c) if c in CLASSES else -1 for c in df.classe]
        self.transformation = transformation
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        image = Image.open(RACINE / self.chemins[i]).convert('RGB')
        return self.transformation(image), self.etiquettes[i]

charge_entrainement = DataLoader(Feuilles(entrainement, aug_entrainement), TAILLE_LOT, shuffle=True, num_workers=2)
charge_validation = DataLoader(Feuilles(validation, aug_evaluation), TAILLE_LOT, num_workers=2)
charge_test = DataLoader(Feuilles(test, aug_evaluation), TAILLE_LOT, num_workers=2)

In [ ]:
# 5. Modèle : MobileNetV3-Large pré-entraîné sur ImageNet, dernière couche remplacée (3 classes)
#    Choisi parce qu'il est léger : l'énoncé demande une inférence possible sans GPU.
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

modele = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
modele.classifier[3] = torch.nn.Linear(modele.classifier[3].in_features, len(CLASSES))
modele = modele.to(appareil)
print(f'{sum(p.numel() for p in modele.parameters()) / 1e6:.1f} millions de paramètres')

In [ ]:
# 6. Entraînement ; on garde l'époque qui a la meilleure F1 moyenne en validation
from sklearn.metrics import f1_score

def predire(charge):
    modele.eval(); probas, vraies = [], []
    with torch.no_grad():
        for x, y in charge:
            probas.append(torch.softmax(modele(x.to(appareil)), 1).cpu()); vraies.append(y)
    return torch.cat(probas).numpy(), torch.cat(vraies).numpy()

optimiseur = torch.optim.AdamW(modele.parameters(), lr=TAUX_APPRENTISSAGE, weight_decay=1e-4)
planning = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=EPOQUES)
perte = torch.nn.CrossEntropyLoss()
meilleure_f1, historique = -1, []

for epoque in range(1, EPOQUES + 1):
    modele.train(); debut = time.time(); total = 0
    for x, y in charge_entrainement:
        x, y = x.to(appareil), y.to(appareil)
        optimiseur.zero_grad(); l = perte(modele(x), y); l.backward(); optimiseur.step()
        total += l.item() * len(y)
    planning.step()
    probas, vraies = predire(charge_validation)
    f1 = f1_score(vraies, probas.argmax(1), average='macro')
    historique.append({'epoque': epoque, 'perte': total / len(charge_entrainement.dataset), 'f1_validation': f1})
    print(f'époque {epoque}  perte {historique[-1]["perte"]:.3f}  F1 validation {f1:.3f}  ({time.time() - debut:.0f} s)')
    if f1 > meilleure_f1:
        meilleure_f1 = f1
        torch.save(modele.state_dict(), '/content/maladies_mobilenetv3.pt')

modele.load_state_dict(torch.load('/content/maladies_mobilenetv3.pt'))
print('Meilleure F1 en validation :', round(meilleure_f1, 3))

In [ ]:
# 7. Seuil « inconnu » : réglé sur la validation seulement (aucune image du test n'est utilisée)
#    Si la confiance maximale du modèle est sous le seuil, il répond « inconnu ».
probas_val, vraies_val = predire(charge_validation)
SEUIL = float(np.quantile(probas_val.max(1), 1 - ACCEPTATION_VALIDATION))
print(f'Seuil de confiance : {SEUIL:.3f}  (garde {ACCEPTATION_VALIDATION:.0%} des images de validation)')

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(vraies_val, probas_val.argmax(1), target_names=CLASSES, digits=3))

In [ ]:
# 8. Test sur le Bangladesh : utilisé une seule fois, après tous les réglages
probas_test, _ = predire(charge_test)
reponse = np.where(probas_test.max(1) >= SEUIL, np.array(CLASSES)[probas_test.argmax(1)], 'inconnu')
t = test.assign(reponse=reponse, confiance=probas_test.max(1))

# Réponse attendue : la sigatoka du Bangladesh compte comme Black Sigatoka (forme non précisée par le jeu)
attendu = {'saine': 'saine', 'sigatoka': 'black_sigatoka', 'cordana': 'inconnu', 'pestalotiopsis': 'inconnu'}
t['attendu'] = t.classe.map(attendu)

print('Matrice de confusion (lignes : classe réelle, colonnes : réponse du modèle)')
print(pd.crosstab(t.classe, t.reponse))

connues = t[t.classe.isin(['saine', 'sigatoka'])]
inconnues = t[t.classe.isin(['cordana', 'pestalotiopsis'])]
saines = t[t.classe == 'saine']
malade_dit = t[t.reponse.isin(['black_sigatoka', 'fusarium'])]
resultats = {
    'exactitude_classes_connues': float((connues.reponse == connues.attendu).mean()),
    'rappel_sigatoka': float((t[t.classe == 'sigatoka'].reponse == 'black_sigatoka').mean()),
    'precision_malade': float(malade_dit.classe.isin(['sigatoka']).mean()) if len(malade_dit) else None,
    'fausses_alertes_sur_saines': float(saines.reponse.isin(['black_sigatoka', 'fusarium']).mean()),
    'rejet_inconnu': float((inconnues.reponse == 'inconnu').mean()),
    'rejet_a_tort_classes_connues': float((connues.reponse == 'inconnu').mean()),
}
from sklearn.metrics import roc_auc_score
# Capacité à séparer connu / inconnu, indépendamment du seuil (1 = parfait, 0,5 = hasard)
resultats['auroc_inconnu'] = float(roc_auc_score(t.classe.isin(['cordana', 'pestalotiopsis']), -t.confiance))
for k, v in resultats.items():
    print(f'{k:32s} {v:.3f}' if v is not None else f'{k:32s} -')

In [ ]:
# 9. Sauvegarde sur le Drive : poids, seuil, résultats, réponses image par image
import shutil, os
sortie = Path(DOSSIER_DRIVE) / 'resultats_maladies'
sortie.mkdir(exist_ok=True)
shutil.copy('/content/maladies_mobilenetv3.pt', sortie)
json.dump({'classes': CLASSES, 'seuil_inconnu': SEUIL, 'epoques': EPOQUES, 'historique': historique,
           'meilleure_f1_validation': meilleure_f1, 'test_bangladesh': resultats,
           'taille_poids_mo': os.path.getsize(sortie / 'maladies_mobilenetv3.pt') / 1e6},
          open(sortie / 'resultats.json', 'w'), indent=2, ensure_ascii=False)
t.to_csv(sortie / 'reponses_test.csv', index=False)
print('Écrit dans', sortie)